# Lab | Web Scraping

¡Bienvenido al laboratorio de aventuras de web scraping "Books to Scrape"!

**Objetivo**

En este laboratorio, nos embarcaremos en una misión para descubrir información valiosa a partir de los datos disponibles en Books to Scrape, una plataforma en línea que muestra una amplia variedad de libros. Como analista de datos, se le ha asignado la tarea de extraer un subconjunto específico de datos de libros de Books to Scrape para ayudar a las editoriales a comprender el panorama de libros altamente calificados en diferentes géneros. Sus conocimientos ayudarán a dar forma a futuras estrategias de marketing de libros y decisiones de publicación.

**Fondo**

En un mundo donde los datos se han convertido en la nueva moneda, las empresas están aprovechando el big data para tomar decisiones informadas que impulsen el éxito y la rentabilidad. La industria editorial, al igual que otras, utiliza análisis de datos para comprender las tendencias del mercado, las preferencias de los lectores y el rendimiento de los libros en función de factores como el género, el autor y las calificaciones. Books to Scrape sirve como una rica fuente de dichos datos, ya que ofrece información detallada sobre una amplia gama de libros, lo que la convierte en una plataforma ideal para extraer conocimientos que ayuden en la toma de decisiones informadas dentro del mundo literario.

**Tarea**

Su tarea es crear una secuencia de comandos de Python utilizando BeautifulSoup y pandas para extraer datos de libros de Books to Scrape, centrándose en las clasificaciones y los géneros de los libros. El guión debería poder filtrar libros con calificaciones superiores a un cierto umbral y en géneros específicos. Además, el script debe estructurar los datos extraídos en un formato tabular utilizando pandas para un análisis más detallado.

**Resultado esperado**

Una función llamada `scrape_books` que toma dos parámetros: `min_rating` y `max_price`. La función debe extraer los datos del libro del sitio web "Books to Scrape" y devolver un DataFrame `pandas` con las siguientes columnas:

**Resultado esperado**

- Una función llamada `scrape_books` que toma dos parámetros: `min_rating` y `max_price`.
- La función debería devolver un DataFrame con las siguientes columnas:
  - **UPC**: El Código Universal de Producto (UPC) del libro.
  - **Título**: El título del libro.
  - **Precio (£)**: El precio del libro en libras.
  - **Calificación**: La calificación del libro (1-5 estrellas).
  - **Género**: El género del libro.
  - **Disponibilidad**: Si el libro está en stock o no.
  - **Descripción**: Una breve descripción o descripción del producto del libro (si está disponible).
  
Ejecutará este script para extraer datos de libros con una calificación mínima de "4.0 y superior" y un precio máximo de "£20". 

Recuerde experimentar con diferentes calificaciones y precios para asegurarse de que su código sea versátil y pueda manejar varias búsquedas de manera efectiva.

**Recursos**

- [Documentación de Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Documentación de Pandas](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Libros para raspar] (https://books.toscrape.com/)

**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

In [12]:
def scrape_books(min_rating, max_price):
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    books_data = []

    # Recorrer las 50 páginas del catálogo
    for page_num in range(1, 51):
        current_page_url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
        
        try:
            response = requests.get(current_page_url, timeout=10)
        except:
            break 

        soup = BeautifulSoup(response.content, 'html.parser')
        products = soup.find_all('article', class_='product_pod')
            
        for product in products:
            # Extraemos las clases y tomamos directamente la segunda (la palabra del número)
            star_class = product.find('p', class_='star-rating')['class']
            rating_text = star_class[1] 
            rating = rating_map.get(rating_text, 0)
            
            if rating < min_rating:
                continue 

            # Limpiamos el precio quitando cualquier símbolo que no sea número o punto
            price_text = product.find('p', class_='price_color').text
            price = float(re.sub(r'[^\d.]', '', price_text))
            
            if price > max_price:
                continue

            # Conseguimos el título y armamos la URL completa del detalle del libro
            title = product.find('h3').find('a')['title']
            detail_rel_url = product.find('h3').find('a')['href']
            detail_url = urljoin("https://books.toscrape.com/catalogue/", detail_rel_url)
            
            try:
                detail_response = requests.get(detail_url, timeout=10)
            except: 
                continue  

            detail_soup = BeautifulSoup(detail_response.content, 'html.parser')
            
            # Buscamos el género dentro del menú de migas de pan (breadcrumb)
            breadcrumb = detail_soup.find('ul', class_='breadcrumb')
            if breadcrumb:
                lis = breadcrumb.find_all('li')
                # Corregido: accedemos al índice 2 antes de pedir el texto
                genre = lis[2].text.strip() if len(lis) > 2 else "Desconocido"
            else:
                genre = "Desconocido"
                
            # Extraemos la tabla de información técnica (UPC y disponibilidad)
            table = detail_soup.find('table', class_='table-striped')
            upc = ""
            availability = ""
            
            if table:
                rows = table.find_all('tr')
                for row in rows:
                    header = row.find('th').text.strip()
                    value = row.find('td').text.strip()
                    if header == "UPC":
                        upc = value
                    elif header == "Availability":
                        availability = value
            
            # Buscamos la descripción que está justo después del título de descripción
            desc_heading = detail_soup.find('div', id='product_description')
            if desc_heading:
                description = desc_heading.find_next_sibling('p').text.strip()
            else:
                description = "Sin descripción"
                
            # Guardamos todo en el diccionario
            books_data.append({
                "UPC": upc,
                "Título": title,
                "Precio (£)": price,
                "Calificación": rating,
                "Género": genre,
                "Disponibilidad": availability,
                "Descripción": description
            })
            
    df = pd.DataFrame(books_data)
    return df

In [13]:
df_filtrado = scrape_books(min_rating=4.0, max_price=20.0)

In [14]:
df_filtrado

,UPC,Título,Precio (£),Calificación,Género,Disponibilidad,Descripción
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
...,...,...,...,...,...,...,...
70,9c96cd1329fbd82d,The Zombie Room,19.69,5,Default,In stock (1 available),An unlikely bond is forged between three men f...
71,b78deb463531d078,The Silent Wife,12.34,5,Fiction,In stock (1 available),A chilling psychological thriller about a marr...
72,4280ac3eab57aa5d,The Girl You Lost,12.29,5,Mystery,In stock (1 available),Eighteen years ago your baby daughter was snat...
73,29fc016c459aeb14,The Edge of Reason (Bridget Jones #2),19.18,4,Womens Fiction,In stock (1 available),Monday 27 January“7:15 a.m. Hurrah! The wilder...
